# 01 — Exploração e Pré-processamento

Reprodução do IDS multicamada SOME/IP (Luo et al. 2023).
Este notebook demonstra a **Fase 1**: carga dos CSVs e pré-processamento da camada de IA
(normalização min-max por Message ID + janelamento len=91/step=30).

Ver `docs/plano-reproducao.md`.

In [ ]:
# Permite importar o pacote `src` a partir da raiz do projeto
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
print('ROOT =', ROOT)

## 1. Carga e exploração

In [ ]:
from src.data import load

df = load.load_ai_csv('bend', 't')  # cenário 'bend', condição tamper
print('shape:', df.shape)
print('Message IDs:', sorted(df['message_id'].unique()))
print('Labels:', df['label'].value_counts().to_dict(), '  # 0=Tamper 1=Normal 2=Replay')
df.head()

In [ ]:
# Sinais por Message ID (quantos signals são válidos em cada um)
load.SIGNALS_PER_ID

## 2. Pré-processamento (normalização + janelamento + split)

In [ ]:
from src.data import preprocess

# Comece com 1 cenário (rápido). Use scenarios=load.SCENARIOS para o dataset completo.
ds = preprocess.build_ai_dataset(scenarios=['bend'])
print(ds.summary())

In [ ]:
# Formato pronto para o multi-GRU:
#   X: (n_seq, 91, 6)  sinais normalizados
#   M: (n_seq, 91)     indice do Message ID por pacote (rotear cada pacote p/ sua GRU)
#   y: (n_seq,)        label da janela
print('X_train:', ds.X_train.shape)
print('M_train:', ds.M_train.shape, '-> valores:', sorted(set(ds.M_train.ravel().tolist())))
print('y_train:', ds.y_train.shape)
print('ID_TO_IDX:', load.ID_TO_IDX)

## Próximo passo

Fase 2/3: implementar a Camada 1 (regras) em `src/rules/` e o multi-GRU em `src/models/`.
O `ds` acima já entrega os tensores no formato esperado pelo modelo.